# F1 Pit Stop Trends and Weather Strategy

Explore recorded pit-stop patterns from 2011 onward and race-start trackside weather from 2023 onward. These are descriptive associations—not causal claims—and the 2026 season may be incomplete.


In [ ]:
import os
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

DATA_DIR = Path(os.getenv("F1_DATA_DIR", "/kaggle/input/formula-1-pit-stop-dataset"))
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Dataset directory not found: {DATA_DIR}")
print(f"Reading data from {DATA_DIR}")


In [ ]:
pits = pd.read_csv(DATA_DIR / "pit_events.csv")
drivers = pd.read_csv(DATA_DIR / "race_drivers.csv")
context = pd.read_csv(DATA_DIR / "race_context.csv")
race_keys = ["season", "round_number"]

entrants = drivers.groupby(race_keys).size().rename("entrants")
race_pits = pits.groupby(race_keys).size().rename("pit_stops")
by_race = context.merge(entrants, on=race_keys, how="left").merge(race_pits, on=race_keys, how="left")
by_race["pit_stops"] = by_race["pit_stops"].fillna(0)
by_race["stops_per_driver"] = by_race["pit_stops"] / by_race["entrants"]
by_race = by_race[by_race["season"] >= 2011].copy()
by_race.head()


## How pit-stop frequency changed


In [ ]:
annual = by_race.groupby("season").agg(
    races=("round_number", "size"), mean_stops_per_driver=("stops_per_driver", "mean")
).reset_index()
plt.figure(figsize=(11, 4))
sns.lineplot(data=annual, x="season", y="mean_stops_per_driver", marker="o", color="#e10600")
plt.title("Average recorded stops per driver and race")
plt.ylabel("Stops per driver")
plt.tight_layout()
annual.tail(10)


## Typical pit laps and durations

`pit_duration_s` is source-defined pit-lane duration and is not interchangeable with stationary `stop_duration_s`.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(pits["lap_number"].dropna(), bins=35, ax=axes[0], color="#3671c6")
axes[0].set_title("Recorded pit-stop lap distribution")
duration = pits["pit_duration_s"].dropna()
duration = duration[duration.between(duration.quantile(.01), duration.quantile(.99))]
sns.histplot(duration, bins=35, ax=axes[1], color="#ff8700")
axes[1].set_title("Pit-lane duration (1st–99th percentile)")
plt.tight_layout()


## Race-start rain and stopping frequency

Weather is measured near the scheduled start and does not describe every lap. It is suitable for a lights-out snapshot, not a full-race weather history.


In [ ]:
modern = by_race[by_race["start_rainfall"].notna()].copy()
modern["start_condition"] = np.where(modern["start_rainfall"].astype(float).gt(0), "Rain at start", "Dry at start")
display(modern.groupby("start_condition")["stops_per_driver"].agg(["count", "mean", "median"]))
plt.figure(figsize=(7, 4))
sns.boxplot(data=modern, x="start_condition", y="stops_per_driver", palette=["#3671c6", "#e10600"])
plt.title("Stops per driver by race-start rainfall")
plt.xlabel("")
plt.tight_layout()


## Most stop-intensive races


In [ ]:
cols = ["season", "round_number", "circuit_short_name", "country_name", "pit_stops", "entrants", "stops_per_driver", "start_rainfall"]
by_race.sort_values("stops_per_driver", ascending=False)[cols].head(15)
